In [ ]:
# EXEMPLO 3 (NOVO) — REFORÇO (Q-LEARNING) PARA TRIAGEM FORENSE DE ALERTAS
# Objetivo didático: fazer o agente aprender as 3 ações:
# 1) INVESTIGAR  2) FILA  3) IGNORAR
#
# Contexto: SOC/Equipe forense recebe alertas e precisa decidir o que fazer.
# Estados:
# - risco: 0=baixo, 1=médio, 2=alto
# - pressão: 0=baixa, 1=alta (muitos alertas chegando -> tempo “vale mais”)
#
# Estratégia do ambiente (recompensas) para induzir as 3 ações:
# - risco alto: ignorar é muito caro; investigar tende a ser melhor.
# - risco médio: "fila" costuma ser melhor (investigar agora custa, ignorar arrisca).
# - risco baixo: ignorar tende a ser melhor (investigar é desperdício).
#
# Além disso, a pressão alta aumenta o custo de investigar e também penaliza fila
# (porque atrasar em pressão alta piora o SLA).

import numpy as np

rng = np.random.default_rng(123)

# ------------------------------
# Estados e ações
# ------------------------------
def state_to_idx(risk_level, time_pressure):
    return risk_level * 2 + time_pressure  # 6 estados

n_states = 6
actions = ["investigar", "fila", "ignorar"]
n_actions = len(actions)

Q = np.zeros((n_states, n_actions), dtype=float)

# ------------------------------
# Hiperparâmetros (podem ser ajustados em aula)
# ------------------------------
alpha = 0.20
gamma = 0.90
eps = 0.20
episodes = 6000
steps_per_episode = 15

# ------------------------------
# Ambiente: geração de alertas
# ------------------------------
def sample_alert():
    # mais alertas de risco baixo e médio; risco alto é mais raro
    risk_level = rng.choice([0, 1, 2], p=[0.55, 0.30, 0.15])
    # pressão alta ocorre com frequência moderada
    time_pressure = rng.choice([0, 1], p=[0.60, 0.40])
    return risk_level, time_pressure

def reward_function(risk_level, time_pressure, action):
    """
    Desenho de recompensas para o agente aprender as 3 ações:

    - A probabilidade de incidente cresce com o risco.
    - Investigar tem custo (maior em pressão alta).
    - Fila tem custo moderado e "perde valor" em pressão alta (SLA).
    - Ignorar ganha um pequeno bônus (economiza tempo), mas é fortemente punido se era incidente.
    """
    # Probabilidade de incidente por risco (mais separação para ficar didático)
    p_incident = {0: 0.02, 1: 0.20, 2: 0.70}[risk_level]
    incident = (rng.random() < p_incident)

    # custos variam com pressão
    investigate_cost = 2.0 if time_pressure == 0 else 3.5
    queue_cost = 0.8 if time_pressure == 0 else 1.6

    if action == "investigar":
        # Se for incidente: grande recompensa (captura ataque)
        # Se for falso alarme: penaliza custo de investigação
        return (10.0 - investigate_cost) if incident else (-1.0 - investigate_cost)

    if action == "fila":
        # Fila é "meio termo": não resolve agora, mas reduz risco de ignorar.
        # Se era incidente: recompensa pequena (vai ser visto depois) menos custo/atraso
        # Se não era incidente: penaliza pouco (gastou triagem/tempo)
        return (2.5 - queue_cost) if incident else (-0.3 - queue_cost)

    if action == "ignorar":
        # Ignorar economiza tempo se não for incidente
        # Se for incidente: punição forte
        return (0.8 if time_pressure == 0 else 0.4) if not incident else (-12.0)

    raise ValueError("Ação inválida")

# ------------------------------
# Treinamento Q-learning
# ------------------------------
for ep in range(episodes):
    for _ in range(steps_per_episode):
        risk_level, time_pressure = sample_alert()
        s = state_to_idx(risk_level, time_pressure)

        # ε-greedy
        if rng.random() < eps:
            a = rng.integers(0, n_actions)
        else:
            a = int(np.argmax(Q[s]))

        r = reward_function(risk_level, time_pressure, actions[a])

        # próximo estado: chega um novo alerta
        risk2, press2 = sample_alert()
        s2 = state_to_idx(risk2, press2)

        Q[s, a] = Q[s, a] + alpha * (r + gamma * np.max(Q[s2]) - Q[s, a])

# ------------------------------
# Política aprendida (deve usar as 3 ações em estados diferentes)
# ------------------------------
print("Política aprendida por estado:")
for risk_level in [0, 1, 2]:
    for time_pressure in [0, 1]:
        s = state_to_idx(risk_level, time_pressure)
        best_a = int(np.argmax(Q[s]))
        print(f"  risco={risk_level} pressao={time_pressure} -> {actions[best_a]}")

print("\nQ-table (valores aproximados):")
print(np.round(Q, 2))

# ------------------------------
# Modo interativo via input (bom para aula)
# ------------------------------
print("\n=== SIMULADOR DE TRIAGEM (ENTRADA MANUAL) ===")
print("Risco: 0=baixo, 1=médio, 2=alto | Pressão: 0=baixa, 1=alta")

while True:
    try:
        risk_level = int(input("\nInforme o risco do alerta (0/1/2): ").strip())
        time_pressure = int(input("Informe a pressão do SOC (0/1): ").strip())
    except ValueError:
        print("Entrada inválida. Digite apenas números inteiros.")
        continue

    if risk_level not in (0, 1, 2) or time_pressure not in (0, 1):
        print("Valores fora do intervalo. Tente novamente.")
        continue

    s = state_to_idx(risk_level, time_pressure)
    best_action = actions[int(np.argmax(Q[s]))]

    # Exibir também os Q-values para discussão em sala
    qvals = Q[s]
    print("\nRecomendação do agente:", best_action.upper())
    print(f"Q-values -> investigar={qvals[0]:.2f} | fila={qvals[1]:.2f} | ignorar={qvals[2]:.2f}")

    cont = input("\nTestar outro alerta? (s/n): ").strip().lower()
    if cont not in ("s", "sim", "y", "yes"):
        print("Encerrando.")
        break


Política aprendida por estado:
  risco=0 pressao=0 -> ignorar
  risco=0 pressao=1 -> fila
  risco=1 pressao=0 -> fila
  risco=1 pressao=1 -> investigar
  risco=2 pressao=0 -> fila
  risco=2 pressao=1 -> investigar

Q-table (valores aproximados):
[[-1.14  1.14  1.89]
 [-1.71  0.5  -0.79]
 [ 0.14  0.89 -3.15]
 [-0.25 -0.55 -2.01]
 [ 3.3   4.61 -4.28]
 [ 4.63  3.03 -7.71]]

=== SIMULADOR DE TRIAGEM (ENTRADA MANUAL) ===
Risco: 0=baixo, 1=médio, 2=alto | Pressão: 0=baixa, 1=alta

Informe o risco do alerta (0/1/2): 2
Informe a pressão do SOC (0/1): 1

Recomendação do agente: INVESTIGAR
Q-values -> investigar=4.63 | fila=3.03 | ignorar=-7.71
